# DTCR-U-Net: Kaggle Training & Evaluation Pipeline
This notebook trains the **Dual-Task Consistency Regularized U-Net (DTCR-U-Net)** on Kaggle GPU.

### Step 1: Ensure Kaggle GPU is Enabled
Go to **Settings** (right-hand panel) -> **Accelerator** -> **GPU T4 x2** or **P100**.

In [ ]:
# Clone your GitHub repo (Replace with your GitHub repository URL if needed)
!git clone https://github.com/shadyOg/fyp-dtcr-unet.git
%cd fyp-dtcr-unet/dtcr-unet
!pip install -r requirements.txt

### Step 2: Run Data Preprocessing
Extracts 2D slices, performs HU windowing, CLAHE, Level Set Map computation ($T(y)$), and creates patient-level train/val/test splits.

In [ ]:
import os

# Check mounted Kaggle inputs
print("Mounted inputs in /kaggle/input:", os.listdir('/kaggle/input'))

# Set dataset paths based on your added inputs
MOSMED_PATH = '/kaggle/input/mosmeddata-chest-ct-scans-with-covid19'
PROCESSED_OUT = '/kaggle/working/data/processed'

!python data/preprocessing.py \
    --mosmed {MOSMED_PATH} \
    --out {PROCESSED_OUT} \
    --size 256 \
    --context 1 \
    --labeled-ratio 0.4

### Step 3: Train DTCR-U-Net (Dual-Task Semi-Supervised)
Runs 80 epochs with AdamW, Cosine Annealing scheduler, dynamic consistency weight $\lambda_d(t)$, and automatic checkpointing to `/kaggle/working/checkpoints/`.

In [ ]:
!python train.py \
    --data {PROCESSED_OUT} \
    --epochs 80 \
    --batch-size 16 \
    --lr 0.001 \
    --labeled-ratio 0.4 \
    --checkpoints /kaggle/working/checkpoints \
    --outputs /kaggle/working/outputs

### Step 4: Evaluate Model on Test Set
Computes Dice, Sensitivity (SE), Specificity (SP), Accuracy, F1, and Hausdorff Distance (HD) on the test set, and saves visual segmentation comparisons.

In [ ]:
!python evaluate.py \
    --checkpoint /kaggle/working/checkpoints/best_model.pth \
    --data {PROCESSED_OUT} \
    --out /kaggle/working/outputs/evaluation \
    --batch-size 16